In [2]:
import pandas as pd

models = ['claude3', 'google', 'gpt35']
df = {}

for model in models:
    df[model] = pd.read_csv(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/growing_dict/final_terms_{model}.csv")
    

df_human = {}
tgt_langs = [
    "Chinese",
    "Arabic",
    "French",
    "Japanese",
    "Russian",
]

for tgt_lang in tgt_langs:
    df_human[tgt_lang] = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_final/" + tgt_lang + ".csv")
    df_human[tgt_lang]['English_human'] = df_human[tgt_lang]['English']
    df_human[tgt_lang]['English'] = df_human[tgt_lang]['English'].str.lower()

In [3]:
info_total = {}

dfs_merged ={}

for tgt_lang in tgt_langs:

    # Align the dataframes based on the "English" column using an inner join
    df_merged = df['claude3'].merge(df['google'], on="English", suffixes=('_claude', '_google'))\
                        .merge(df['gpt35'], on="English", suffixes=('', '_gpt35'))\
                        .merge(df_human[tgt_lang], on="English", suffixes=('', '_final'))

    col_claude = f"{tgt_lang}_claude"
    col_google = f"{tgt_lang}_google"
    col_gpt35 = f"{tgt_lang}"
    col_human = f"{tgt_lang}_final"
        
    
    # print(df_merged.columns)
    # print(df_merged.shape)
    
    
    info = {
        "claude_vs_human": sum(df_merged[col_claude] == df_merged[col_human]),
        "google_vs_human": sum(df_merged[col_google] == df_merged[col_human]),
        "gpt35_vs_human": sum(df_merged[col_gpt35] == df_merged[col_human]),
        "total": df_merged.shape[0]
    }
    
    info_total[tgt_lang] = info
    
    dfs_merged[tgt_lang] = df_merged

info_total


{'Chinese': {'claude_vs_human': 2835,
  'google_vs_human': 2572,
  'gpt35_vs_human': 2423,
  'total': 4093},
 'Arabic': {'claude_vs_human': 1197,
  'google_vs_human': 1491,
  'gpt35_vs_human': 943,
  'total': 4091},
 'French': {'claude_vs_human': 2335,
  'google_vs_human': 2105,
  'gpt35_vs_human': 580,
  'total': 4092},
 'Japanese': {'claude_vs_human': 2356,
  'google_vs_human': 2031,
  'gpt35_vs_human': 1394,
  'total': 4091},
 'Russian': {'claude_vs_human': 1625,
  'google_vs_human': 1744,
  'gpt35_vs_human': 1155,
  'total': 4091}}

In [20]:
# Convert dictionary to DataFrame
df_res = pd.DataFrame.from_dict(info_total, orient='index')
df_res

,claude_vs_human,google_vs_human,gpt35_vs_human,total
Chinese,2835,2572,2423,4093
Arabic,1197,1491,943,4091
French,2335,2105,580,4092
Japanese,2356,2031,1394,4091
Russian,1625,1744,1155,4091


In [21]:
cols = ['claude_vs_human', 'google_vs_human', 'gpt35_vs_human']
for col in cols:
    df_res[col] /= df_res['total']

df_res = df_res[cols]
df_res = df_res.applymap(lambda x: f"{x:.2%}" if isinstance(x, (float, int)) else x)
print(df_res.to_latex())

/tmp/ipykernel_1258860/3722064180.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_res = df_res.applymap(lambda x: f"{x:.2%}" if isinstance(x, (float, int)) else x)


\begin{tabular}{llll}
\toprule
 & claude_vs_human & google_vs_human & gpt35_vs_human \\
\midrule
Chinese & 69.26% & 62.84% & 59.20% \\
Arabic & 29.26% & 36.45% & 23.05% \\
French & 57.06% & 51.44% & 14.17% \\
Japanese & 57.59% & 49.65% & 34.07% \\
Russian & 39.72% & 42.63% & 28.23% \\
\bottomrule
\end{tabular}



# store English, English_human

In [4]:
for tgt_lang in tgt_langs:
    dfs_merged[tgt_lang][['English', 'English_human']].to_csv(f"intersected_term_to_translate_{tgt_lang}.csv")

# check GPT-4o, GPT-4o-mini translation quality

In [5]:
import pandas as pd
import json

models = ['gpt-4o', 'gpt-4o-mini']
df_model = {
    'gpt-4o': {},
    'gpt-4o-mini': {}
}
df_human = {}

tgt_langs = [
    "Chinese",
    "Arabic",
    "French",
    "Japanese",
    "Russian",
]

for model in models:
    for tgt_lang in tgt_langs:
        df_model[model][tgt_lang] = pd.DataFrame([json.loads(i) for i in open(f'translations/{model}_{tgt_lang}.jsonl', 'r').readlines()])
        df_model[model][tgt_lang] = df_model[model][tgt_lang].drop_duplicates(subset="English")

for tgt_lang in tgt_langs:
    df_human[tgt_lang] = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_final/" + tgt_lang + ".csv")
    df_human[tgt_lang]['English_human'] = df_human[tgt_lang]['English']
    df_human[tgt_lang]['English'] = df_human[tgt_lang]['English'].str.lower()

In [7]:
info_total = {}

dfs_merged ={}

for tgt_lang in tgt_langs:
    info = {}
    
    for model in models:

        df_merged = df_human[tgt_lang].merge(df_model[model][tgt_lang], on='English', suffixes=("_final", f"_{model}"))
        
        # print(df_merged.columns)
        print(df_merged.shape)
        # print(df_merged)
        
        
        info[f"{model}_vs_human"] = sum(df_merged['translation'] == df_merged[tgt_lang]) / df_merged.shape[0]
                
    info_total[tgt_lang] = info
    
    dfs_merged[tgt_lang] = df_merged

info_total

(500, 6)
(4093, 6)
(500, 6)
(4091, 6)
(500, 6)
(4092, 6)
(500, 6)
(4091, 6)
(500, 6)
(4091, 6)


{'Chinese': {'gpt-4o_vs_human': 0.768,
  'gpt-4o-mini_vs_human': 0.7478622037625213},
 'Arabic': {'gpt-4o_vs_human': 0.398,
  'gpt-4o-mini_vs_human': 0.3933023710584209},
 'French': {'gpt-4o_vs_human': 0.582,
  'gpt-4o-mini_vs_human': 0.6344086021505376},
 'Japanese': {'gpt-4o_vs_human': 0.678,
  'gpt-4o-mini_vs_human': 0.7394280127108287},
 'Russian': {'gpt-4o_vs_human': 0.412,
  'gpt-4o-mini_vs_human': 0.48252261060865315}}